# Quick Order-only Checkpoint Eval

지정한 LGT/direct-order checkpoint만 validation 100개에서 빠르게 full-order 평가합니다.


In [ ]:
# 1) Install dependencies, then restart runtime once
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_quick_order_eval_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + locate existing run
from google.colab import drive
drive.mount("/content/drive")

import ast
import gc
import glob
import json
import os
import random
import re
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

RUN_ID = "20260712_234828"
LGT_ROOT_CANDIDATES = [
    "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_direct_order_multitask_v1",
    "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_multitask_v1",
]

CHECKPOINT_STEPS = [2000, 2250, 2500, 2750, 3000]
ORDER_EVAL_ROWS = 100

SEED = 42
VALID_RATIO = 0.1
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR

def resolve_existing_output_dir(run_id):
    checked = []
    for root in LGT_ROOT_CANDIDATES:
        candidate = os.path.join(root, "runs", run_id, "lgt_multitask")
        checked.append(candidate)
        if os.path.isdir(candidate):
            return root, candidate
    raise RuntimeError("Could not find run. Checked:\n" + "\n".join(checked))

LGT_ROOT, OUTPUT_DIR = resolve_existing_output_dir(RUN_ID)
EVAL_DIR = os.path.join(OUTPUT_DIR, "quick_order_eval")
os.makedirs(EVAL_DIR, exist_ok=True)

CHECKPOINT_DIRS = [os.path.join(OUTPUT_DIR, f"checkpoint-{step}") for step in CHECKPOINT_STEPS]
missing = [path for path in CHECKPOINT_DIRS if not os.path.exists(os.path.join(path, "adapter_config.json"))]
if missing:
    raise RuntimeError("Missing adapter checkpoint(s):\n" + "\n".join(missing))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("output dir:", OUTPUT_DIR)
print("eval dir:", EVAL_DIR)
print("checkpoints:")
for path in CHECKPOINT_DIRS:
    print(" ", path)

In [ ]:
# 3) Data split + order prompt helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result

def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]

def format_order(answer):
    return "[" + ", ".join(str(int(value)) for value in answer) + "]"

def parse_order_prediction(text):
    match = re.fullmatch(
        r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*",
        str(text),
    )
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None

def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()

def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]

def task_instruction(sentence):
    return (
        f"Caption:\n{sentence}\n\n"
        "Question: Arrange all images in chronological order.\n"
        "Return only one Python-style list of image numbers, such as [1, 2, 3, 4].\n"
        "Do not output any explanation."
    )

def make_order_messages(example):
    content = []
    for input_number in range(1, 5):
        content.append({"type": "text", "text": f"\nImage {input_number}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]

def make_order_example(row):
    sample_id = str(row["Id"])
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    answer = [int(value) for value in row["Answer_list"]]
    return {
        "Id": sample_id,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
        "instruction": task_instruction(sentence),
        "answer_list": answer,
        "answer_order": order_to_sequence(answer),
    }

PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

def relation_pair_accuracy(pred_order_inputs, answer_ranks):
    if pred_order_inputs is None:
        return 0, len(PAIR_INDICES)
    predicted_ranks = [0] * 4
    for position, input_number in enumerate(pred_order_inputs, start=1):
        predicted_ranks[input_number - 1] = position
    correct = 0
    for first_index, second_index in PAIR_INDICES:
        pred_first_earlier = predicted_ranks[first_index] < predicted_ranks[second_index]
        gold_first_earlier = answer_ranks[first_index] < answer_ranks[second_index]
        correct += int(pred_first_earlier == gold_first_earlier)
    return correct, len(PAIR_INDICES)

train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
eval_df = validation_df.iloc[:ORDER_EVAL_ROWS].copy().reset_index(drop=True)

print("validation rows:", len(validation_df))
print("quick eval rows:", len(eval_df))
display(eval_df[["Id", "Sentence", "Answer_list"]].head())

In [ ]:
# 4) Model load + generation + order-only evaluation
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def disable_sampling_warnings(model):
    generation_config = getattr(model, "generation_config", None)
    if generation_config is None:
        return
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None

def load_adapter_for_eval(adapter_dir):
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    model.config.use_cache = True
    disable_sampling_warnings(model)
    return model

@torch.no_grad()
def generate_order(model, example, max_new_tokens=24):
    text = processor.apply_chat_template(make_order_messages(example), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    output_ids = generated_ids[0, inputs.input_ids.shape[1]:]
    return processor.decode(output_ids, skip_special_tokens=True).strip()

def summarize_order_rows(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return {"total": 0}
    both = df[df["first_and_last_both_correct"] == 1]
    all_pairs = df[df["all_pair_relations_correct"] == 1]
    return {
        "total": int(len(df)),
        "valid_output_rate": float(df["valid_output"].mean()),
        "position_accuracy": float(df["position_correct"].sum() / df["position_total"].sum()),
        "order_pair_accuracy": float(df["pair_correct"].sum() / df["pair_total"].sum()),
        "exact_match_accuracy": float(df["exact_correct"].mean()),
        "first_accuracy_from_order": float(df["first_correct"].mean()),
        "last_accuracy_from_order": float(df["last_correct"].mean()),
        "first_and_last_both_correct": float(df["first_and_last_both_correct"].mean()),
        "all_pair_relations_correct": float(df["all_pair_relations_correct"].mean()),
        "boundary_error_rate": float(df["boundary_error"].mean()),
        "relation_error_rate": float(df["relation_error"].mean()),
        "exact_match_given_correct_boundaries": float(both["exact_correct"].mean()) if not both.empty else np.nan,
        "exact_match_given_all_pair_relations": float(all_pairs["exact_correct"].mean()) if not all_pairs.empty else np.nan,
    }

@torch.no_grad()
def evaluate_order_checkpoint(adapter_dir, dataframe):
    model = load_adapter_for_eval(adapter_dir)
    rows = []
    try:
        for _, row in tqdm(list(dataframe.iterrows()), total=len(dataframe), desc=os.path.basename(adapter_dir)):
            example = make_order_example(row)
            output_text = generate_order(model, example)
            prediction = parse_order_prediction(output_text)
            answer_order = example["answer_order"]
            valid = prediction is not None
            position_correct = sum(p == a for p, a in zip(prediction, answer_order)) if valid else 0
            pair_correct, pair_total = relation_pair_accuracy(prediction, example["answer_list"])
            first_correct = bool(valid and prediction[0] == answer_order[0])
            last_correct = bool(valid and prediction[-1] == answer_order[-1])
            both_boundaries_correct = bool(first_correct and last_correct)
            all_pair_relations_correct = bool(valid and pair_correct == pair_total)
            rows.append({
                "Id": example["Id"],
                "checkpoint": os.path.basename(adapter_dir),
                "output_text": output_text,
                "Prediction": str(prediction) if prediction else "",
                "Answer": str(answer_order),
                "valid_output": valid,
                "position_correct": position_correct,
                "position_total": 4,
                "exact_correct": int(valid and prediction == answer_order),
                "pair_correct": pair_correct,
                "pair_total": pair_total,
                "first_correct": int(first_correct),
                "last_correct": int(last_correct),
                "first_and_last_both_correct": int(both_boundaries_correct),
                "all_pair_relations_correct": int(all_pair_relations_correct),
                "boundary_error": int(valid and not both_boundaries_correct),
                "relation_error": int(valid and pair_correct < pair_total),
            })
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    pred_df = pd.DataFrame(rows)
    summary = summarize_order_rows(rows)
    summary["checkpoint"] = os.path.basename(adapter_dir)
    summary["adapter_dir"] = adapter_dir
    return pred_df, summary

In [ ]:
# 5) Evaluate selected checkpoints only
summary_rows = []

for adapter_dir in CHECKPOINT_DIRS:
    checkpoint_name = os.path.basename(adapter_dir)
    pred_df, summary = evaluate_order_checkpoint(adapter_dir, eval_df)
    pred_path = os.path.join(EVAL_DIR, f"{checkpoint_name}_order100_predictions.csv")
    pred_df.to_csv(pred_path, index=False)
    summary["prediction_csv"] = pred_path
    summary_rows.append(summary)
    print(checkpoint_name, summary)

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["exact_match_accuracy", "order_pair_accuracy", "position_accuracy", "valid_output_rate"],
    ascending=False,
).reset_index(drop=True)

summary_path = os.path.join(EVAL_DIR, "quick_order_checkpoint_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("saved summary:", summary_path)
display(summary_df)